<a href="https://colab.research.google.com/github/Karthikreddy1010/GenAI_Models/blob/main/VAEs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. VAE Architecture
# ============================================================================

class VAE(nn.Module):
    def __init__(self, latent_dim=20, image_channels=1):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim
        self.image_channels = image_channels

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(image_channels, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
        )

        # Latent space
        self.fc_mu = nn.Linear(256 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(256 * 2 * 2, latent_dim)

        # Decoder
        self.fc_decode = nn.Linear(latent_dim, 256 * 2 * 2)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, image_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        """Encode input to latent space"""
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """Reparameterization trick for sampling"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z):
        """Decode from latent space"""
        h = self.fc_decode(z)
        h = h.view(h.size(0), 256, 2, 2)
        x_recon = self.decoder(h)
        return x_recon

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar, z


# ============================================================================
# 2. Loss Function
# ============================================================================

def vae_loss(x_recon, x, mu, logvar):
    """VAE loss: Reconstruction + KL divergence"""
    # Reconstruction loss (Binary Cross-Entropy)
    reconstruction_loss = nn.functional.binary_cross_entropy(x_recon, x, reduction='sum')

    # KL divergence loss
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    # Total loss
    total_loss = reconstruction_loss + kl_loss
    return total_loss, reconstruction_loss, kl_loss


# ============================================================================
# 3. Training Function
# ============================================================================

def train_vae(model, train_loader, device, epochs=10, learning_rate=1e-3):
    """Train the VAE model"""
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    model.train()

    train_losses = []
    recon_losses = []
    kl_losses = []

    print(f"\n{'='*60}")
    print(f"Training VAE on {device}")
    print(f"{'='*60}\n")

    for epoch in range(epochs):
        total_loss = 0
        total_recon = 0
        total_kl = 0

        for batch_idx, (x, _) in enumerate(train_loader):
            x = x.to(device)

            optimizer.zero_grad()
            x_recon, mu, logvar, z = model(x)
            loss, recon_loss, kl_loss = vae_loss(x_recon, x, mu, logvar)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()

        avg_loss = total_loss / len(train_loader.dataset)
        avg_recon = total_recon / len(train_loader.dataset)
        avg_kl = total_kl / len(train_loader.dataset)

        train_losses.append(avg_loss)
        recon_losses.append(avg_recon)
        kl_losses.append(avg_kl)

        if (epoch + 1) % 2 == 0:
            print(f"Epoch {epoch + 1:3d}/{epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Recon: {avg_recon:.4f} | "
                  f"KL: {avg_kl:.4f}")

    print(f"\n{'='*60}\nTraining Complete!\n{'='*60}\n")
    return train_losses, recon_losses, kl_losses


# ============================================================================
# 4. Data Loading
# ============================================================================

def load_datasets(batch_size=128, image_size=32):
    """Load MNIST dataset"""
    transform = transforms.Compose([
        transforms.Pad(2),  # Pad to 32x32
        transforms.ToTensor(),
    ])

    train_dataset = datasets.MNIST(
        root='./data', train=True, transform=transform, download=True
    )
    test_dataset = datasets.MNIST(
        root='./data', train=False, transform=transform, download=True
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
    )

    return train_loader, test_loader, train_dataset, test_dataset


# ============================================================================
# 5. Visualization Functions
# ============================================================================

def plot_training_loss(train_losses, recon_losses, kl_losses, save_path='training_loss.png'):
    """Plot training loss components"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Total loss
    axes[0].plot(train_losses, linewidth=2, label='Total Loss', color='#2E86AB')
    axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Loss', fontsize=12, fontweight='bold')
    axes[0].set_title('VAE Total Loss', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=10)

    # Loss components
    axes[1].plot(recon_losses, linewidth=2, label='Reconstruction Loss', color='#A23B72')
    axes[1].plot(kl_losses, linewidth=2, label='KL Divergence', color='#F18F01')
    axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Loss', fontsize=12, fontweight='bold')
    axes[1].set_title('Loss Components', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Training loss plot saved to {save_path}")
    plt.close()


def plot_reconstructions(model, test_loader, device, num_samples=8, save_path='reconstructions.png'):
    """Plot original vs reconstructed images"""
    model.eval()

    with torch.no_grad():
        x, _ = next(iter(test_loader))
        x = x[:num_samples].to(device)
        x_recon, _, _, _ = model(x)

    fig, axes = plt.subplots(2, num_samples, figsize=(16, 4))

    for i in range(num_samples):
        # Original
        axes[0, i].imshow(x[i].cpu().squeeze(), cmap='gray')
        axes[0, i].set_title('Original', fontsize=10, fontweight='bold')
        axes[0, i].axis('off')

        # Reconstructed
        axes[1, i].imshow(x_recon[i].cpu().squeeze(), cmap='gray')
        axes[1, i].set_title('Reconstructed', fontsize=10, fontweight='bold')
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Reconstructions saved to {save_path}")
    plt.close()


def plot_latent_space_2d(model, test_loader, device, save_path='latent_2d.png'):
    """Plot 2D latent space (for 2D VAE)"""
    model.eval()

    with torch.no_grad():
        mu_list = []
        labels_list = []

        for x, y in test_loader:
            x = x.to(device)
            mu, _ = model.encode(x)
            mu_list.append(mu.cpu())
            labels_list.append(y)

        mu_all = torch.cat(mu_list)
        labels_all = torch.cat(labels_list)

    if model.latent_dim == 2:
        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(mu_all[:, 0], mu_all[:, 1], c=labels_all,
                           cmap='tab10', alpha=0.7, s=30, edgecolors='black', linewidth=0.5)
        ax.set_xlabel('Latent Dimension 1', fontsize=12, fontweight='bold')
        ax.set_ylabel('Latent Dimension 2', fontsize=12, fontweight='bold')
        ax.set_title('2D Latent Space Representation', fontsize=14, fontweight='bold')
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Digit Class', fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ 2D latent space plot saved to {save_path}")
        plt.close()
    else:
        print(f"⚠ Latent dimension is {model.latent_dim}, not 2. Use t-SNE instead.")


def plot_latent_space_tsne(model, test_loader, device, save_path='latent_tsne.png'):
    """Plot latent space using t-SNE"""
    model.eval()

    with torch.no_grad():
        mu_list = []
        labels_list = []

        for x, y in test_loader:
            x = x.to(device)
            mu, _ = model.encode(x)
            mu_list.append(mu.cpu())
            labels_list.append(y)

        mu_all = torch.cat(mu_list).numpy()
        labels_all = torch.cat(labels_list).numpy()

    print("Computing t-SNE (this may take a moment)...")
    tsne = TSNE(n_components=2, random_state=42, n_iter=1000)
    latent_2d = tsne.fit_transform(mu_all)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(latent_2d[:, 0], latent_2d[:, 1], c=labels_all,
                       cmap='tab10', alpha=0.7, s=30, edgecolors='black', linewidth=0.5)
    ax.set_xlabel('t-SNE 1', fontsize=12, fontweight='bold')
    ax.set_ylabel('t-SNE 2', fontsize=12, fontweight='bold')
    ax.set_title('t-SNE Visualization of Latent Space', fontsize=14, fontweight='bold')
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Digit Class', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ t-SNE latent space plot saved to {save_path}")
    plt.close()


def plot_latent_distribution(model, test_loader, device, save_path='latent_dist.png'):
    """Plot distribution of latent dimensions"""
    model.eval()

    with torch.no_grad():
        mu_list = []

        for x, _ in test_loader:
            x = x.to(device)
            mu, _ = model.encode(x)
            mu_list.append(mu.cpu())

        mu_all = torch.cat(mu_list).numpy()

    # Plot histograms of first 4 dimensions
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    for i in range(min(4, model.latent_dim)):
        axes[i].hist(mu_all[:, i], bins=50, alpha=0.7, color='#2E86AB', edgecolor='black')
        axes[i].set_xlabel(f'Latent Dim {i+1}', fontsize=11, fontweight='bold')
        axes[i].set_ylabel('Frequency', fontsize=11, fontweight='bold')
        axes[i].set_title(f'Distribution of Dimension {i+1}', fontsize=12, fontweight='bold')
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Latent distribution plot saved to {save_path}")
    plt.close()


def plot_generated_samples(model, device, num_samples=16, save_path='generated_samples.png'):
    """Generate and plot samples from latent space"""
    model.eval()

    with torch.no_grad():
        z = torch.randn(num_samples, model.latent_dim).to(device)
        samples = model.decode(z)

    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    axes = axes.flatten()

    for i in range(num_samples):
        axes[i].imshow(samples[i].cpu().squeeze(), cmap='gray')
        axes[i].set_title(f'Sample {i+1}', fontsize=9)
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Generated samples saved to {save_path}")
    plt.close()


# ============================================================================
# 6. Main Execution
# ============================================================================

if __name__ == "__main__":
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"CUDA Version: {torch.version.cuda}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB\n")

    # Hyperparameters
    BATCH_SIZE = 128
    EPOCHS = 25
    LATENT_DIM = 20
    LEARNING_RATE = 1e-3

    # Load datasets
    print("Loading MNIST dataset...")
    train_loader, test_loader, train_dataset, test_dataset = load_datasets(
        batch_size=BATCH_SIZE
    )
    print(f"✓ Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}\n")

    # Initialize model
    model = VAE(latent_dim=LATENT_DIM, image_channels=1).to(device)
    print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

    # Train
    train_losses, recon_losses, kl_losses = train_vae(
        model, train_loader, device, epochs=EPOCHS, learning_rate=LEARNING_RATE
    )

    # Evaluate and visualize
    print("Generating visualizations...\n")
    plot_training_loss(train_losses, recon_losses, kl_losses)
    plot_reconstructions(model, test_loader, device, num_samples=8)
    plot_latent_distribution(model, test_loader, device)
    plot_latent_space_tsne(model, test_loader, device)
    plot_generated_samples(model, device, num_samples=16)

    print("✓ All visualizations saved!")

Using device: cuda
GPU: Tesla T4
CUDA Version: 12.8
Memory: 15.64 GB

Loading MNIST dataset...
✓ Train samples: 60000 | Test samples: 10000

Model Parameters: 1,440,489


Training VAE on cuda

Epoch   2/25 | Loss: 114.0984 | Recon: 96.9578 | KL: 17.1406
Epoch   4/25 | Loss: 102.7884 | Recon: 82.9131 | KL: 19.8753
Epoch   6/25 | Loss: 99.9010 | Recon: 79.5207 | KL: 20.3803
Epoch   8/25 | Loss: 98.3413 | Recon: 77.4984 | KL: 20.8429
Epoch  10/25 | Loss: 97.3785 | Recon: 76.3241 | KL: 21.0544
Epoch  12/25 | Loss: 96.6205 | Recon: 75.4639 | KL: 21.1567
Epoch  14/25 | Loss: 96.0740 | Recon: 74.8144 | KL: 21.2597
Epoch  16/25 | Loss: 95.6127 | Recon: 74.2944 | KL: 21.3183
Epoch  18/25 | Loss: 95.2426 | Recon: 73.9261 | KL: 21.3164
Epoch  20/25 | Loss: 94.9039 | Recon: 73.5501 | KL: 21.3537
Epoch  22/25 | Loss: 94.6852 | Recon: 73.2785 | KL: 21.4067
Epoch  24/25 | Loss: 94.3949 | Recon: 72.9759 | KL: 21.4190

Training Complete!

Generating visualizations...

✓ Training loss plot saved to trai

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# VAE with 2D Latent Space for Clear Visualization
# ============================================================================

class VAE_2D(nn.Module):
    """VAE with 2D latent space for easy visualization"""
    def __init__(self, image_channels=1):
        super(VAE_2D, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(image_channels, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
        )

        # Latent space (2D)
        self.fc_mu = nn.Linear(256 * 2 * 2, 2)
        self.fc_logvar = nn.Linear(256 * 2 * 2, 2)

        # Decoder
        self.fc_decode = nn.Linear(2, 256 * 2 * 2)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, image_channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z):
        h = self.fc_decode(z)
        h = h.view(h.size(0), 256, 2, 2)
        x_recon = self.decoder(h)
        return x_recon

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar, z


def vae_loss(x_recon, x, mu, logvar, beta=1.0):
    """VAE loss with beta coefficient for KL weighting"""
    reconstruction_loss = nn.functional.binary_cross_entropy(x_recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = reconstruction_loss + beta * kl_loss
    return total_loss, reconstruction_loss, kl_loss


def train_vae_2d(model, train_loader, device, epochs=20, learning_rate=1e-3):
    """Train 2D VAE"""
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    model.train()

    train_losses = []
    recon_losses = []
    kl_losses = []

    print(f"\n{'='*70}")
    print(f"Training 2D VAE on {device}")
    print(f"{'='*70}\n")

    for epoch in range(epochs):
        total_loss = 0
        total_recon = 0
        total_kl = 0

        for batch_idx, (x, _) in enumerate(train_loader):
            x = x.to(device)

            optimizer.zero_grad()
            x_recon, mu, logvar, z = model(x)
            loss, recon_loss, kl_loss = vae_loss(x_recon, x, mu, logvar, beta=1.0)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()

        avg_loss = total_loss / len(train_loader.dataset)
        avg_recon = total_recon / len(train_loader.dataset)
        avg_kl = total_kl / len(train_loader.dataset)

        train_losses.append(avg_loss)
        recon_losses.append(avg_recon)
        kl_losses.append(avg_kl)

        if (epoch + 1) % 2 == 0:
            print(f"Epoch {epoch + 1:3d}/{epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Recon: {avg_recon:.4f} | "
                  f"KL: {avg_kl:.4f}")

    print(f"\n{'='*70}\nTraining Complete!\n{'='*70}\n")
    return train_losses, recon_losses, kl_losses


def load_data_2d(batch_size=256):
    """Load MNIST dataset"""
    transform = transforms.Compose([
        transforms.Pad(2),
        transforms.ToTensor(),
    ])

    train_dataset = datasets.MNIST(
        root='./data', train=True, transform=transform, download=True
    )
    test_dataset = datasets.MNIST(
        root='./data', train=False, transform=transform, download=True
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
    )

    return train_loader, test_loader


def plot_2d_latent_space(model, test_loader, device, save_path='latent_space_2d.png'):
    """Plot 2D latent space with digit labels"""
    model.eval()

    with torch.no_grad():
        mu_list = []
        labels_list = []

        for x, y in test_loader:
            x = x.to(device)
            mu, _ = model.encode(x)
            mu_list.append(mu.cpu())
            labels_list.append(y)

        mu_all = torch.cat(mu_list).numpy()
        labels_all = torch.cat(labels_list).numpy()

    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))

    # Color map for digits
    colors = plt.cm.tab10(np.linspace(0, 1, 10))

    # Plot each digit class
    for digit in range(10):
        mask = labels_all == digit
        scatter = ax.scatter(mu_all[mask, 0], mu_all[mask, 1],
                           c=[colors[digit]], s=40, alpha=0.7,
                           label=f'Digit {digit}', edgecolors='black', linewidth=0.5)

    ax.set_xlabel('Latent Dimension 1', fontsize=14, fontweight='bold')
    ax.set_ylabel('Latent Dimension 2', fontsize=14, fontweight='bold')
    ax.set_title('VAE 2D Latent Space - MNIST Digits', fontsize=16, fontweight='bold')
    ax.legend(loc='best', fontsize=10, framealpha=0.95)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ 2D Latent space plot saved to {save_path}")
    plt.close()


def plot_latent_traversal(model, device, save_path='latent_traversal.png'):
    """Generate images by traversing latent space"""
    model.eval()

    # Create a grid of latent points
    n_points = 15
    z_range = 3.0
    z1 = np.linspace(-z_range, z_range, n_points)
    z2 = np.linspace(-z_range, z_range, n_points)

    fig, axes = plt.subplots(n_points, n_points, figsize=(15, 15))

    with torch.no_grad():
        for i, z1_val in enumerate(z1):
            for j, z2_val in enumerate(z2):
                z = torch.tensor([[z1_val, z2_val]], dtype=torch.float32).to(device)
                x_sample = model.decode(z)

                axes[i, j].imshow(x_sample[0].cpu().squeeze(), cmap='gray')
                axes[i, j].axis('off')

    # Add labels
    fig.text(0.5, 0.02, 'Latent Dimension 1', ha='center', fontsize=14, fontweight='bold')
    fig.text(0.02, 0.5, 'Latent Dimension 2', va='center', rotation='vertical',
             fontsize=14, fontweight='bold')
    fig.suptitle('Latent Space Traversal - Generated Digits', fontsize=16, fontweight='bold', y=0.995)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Latent traversal plot saved to {save_path}")
    plt.close()


def plot_reconstruction_quality(model, test_loader, device, save_path='reconstruction_quality.png'):
    """Compare original vs reconstructed with variety of digits"""
    model.eval()

    # Get one batch
    x, y = next(iter(test_loader))
    x = x[:10].to(device)
    y = y[:10]

    with torch.no_grad():
        x_recon, mu, logvar, z = model(x)

    fig, axes = plt.subplots(3, 10, figsize=(18, 6))

    for i in range(10):
        # Original
        axes[0, i].imshow(x[i].cpu().squeeze(), cmap='gray')
        axes[0, i].set_title(f'{y[i]}', fontsize=11, fontweight='bold')
        axes[0, i].axis('off')

        # Reconstructed
        axes[1, i].imshow(x_recon[i].cpu().squeeze(), cmap='gray')
        axes[1, i].axis('off')

        # Latent code
        axes[2, i].text(0.5, 0.5, f'z1: {z[i, 0]:.2f}\nz2: {z[i, 1]:.2f}',
                       ha='center', va='center', fontsize=9,
                       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
        axes[2, i].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Reconstructed', fontsize=12, fontweight='bold')
    axes[2, 0].set_ylabel('Latent Code', fontsize=12, fontweight='bold')

    fig.suptitle('VAE Reconstruction Quality with Latent Codes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Reconstruction quality plot saved to {save_path}")
    plt.close()


def plot_loss_curves(train_losses, recon_losses, kl_losses, save_path='loss_curves.png'):
    """Plot training loss curves"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Total loss
    axes[0].plot(train_losses, linewidth=2.5, label='Total Loss', color='#2E86AB', marker='o', markersize=3)
    axes[0].fill_between(range(len(train_losses)), train_losses, alpha=0.2, color='#2E86AB')
    axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Loss', fontsize=12, fontweight='bold')
    axes[0].set_title('VAE Training Loss', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=10)

    # Components
    axes[1].plot(recon_losses, linewidth=2.5, label='Reconstruction Loss', color='#A23B72', marker='s', markersize=3)
    axes[1].plot(kl_losses, linewidth=2.5, label='KL Divergence', color='#F18F01', marker='^', markersize=3)
    axes[1].fill_between(range(len(recon_losses)), recon_losses, alpha=0.2, color='#A23B72')
    axes[1].fill_between(range(len(kl_losses)), kl_losses, alpha=0.2, color='#F18F01')
    axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Loss', fontsize=12, fontweight='bold')
    axes[1].set_title('Loss Components', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Loss curves saved to {save_path}")
    plt.close()


# ============================================================================
# Main Execution
# ============================================================================

if __name__ == "__main__":
    # Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*70}")
    print(f"VAE with 2D Latent Space Visualization")
    print(f"{'='*70}")
    print(f"Using device: {device}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"CUDA Version: {torch.version.cuda}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Hyperparameters
    BATCH_SIZE = 256
    EPOCHS = 25
    LEARNING_RATE = 1e-3

    # Load data
    print(f"\nLoading MNIST dataset...")
    train_loader, test_loader = load_data_2d(batch_size=BATCH_SIZE)
    print(f"✓ Dataset loaded (Train: 60000, Test: 10000)")

    # Create model
    model = VAE_2D(image_channels=1).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n✓ Model created")
    print(f"  Total parameters: {total_params:,}")

    # Train
    train_losses, recon_losses, kl_losses = train_vae_2d(
        model, train_loader, device, epochs=EPOCHS, learning_rate=LEARNING_RATE
    )

    # Visualizations
    print("Generating visualizations...\n")
    plot_loss_curves(train_losses, recon_losses, kl_losses)
    plot_2d_latent_space(model, test_loader, device)
    plot_reconstruction_quality(model, test_loader, device)
    plot_latent_traversal(model, device)

    print("\n✓ All plots generated successfully!")
    print("\nOutput files:")
    print("  1. loss_curves.png - Training loss curves")
    print("  2. latent_space_2d.png - 2D latent space with digit clusters")
    print("  3. reconstruction_quality.png - Original vs reconstructed images")
    print("  4. latent_traversal.png - Generated digits via latent space traversal")


VAE with 2D Latent Space Visualization
Using device: cuda
GPU: Tesla T4
CUDA Version: 12.8
Memory: 15.64 GB

Loading MNIST dataset...
✓ Dataset loaded (Train: 60000, Test: 10000)

✓ Model created
  Total parameters: 1,385,157

Training 2D VAE on cuda

Epoch   2/25 | Loss: 160.0085 | Recon: 154.8152 | KL: 5.1933
Epoch   4/25 | Loss: 150.9308 | Recon: 145.0146 | KL: 5.9162
Epoch   6/25 | Loss: 147.5986 | Recon: 141.4124 | KL: 6.1862
Epoch   8/25 | Loss: 145.5504 | Recon: 139.1876 | KL: 6.3628
Epoch  10/25 | Loss: 143.9328 | Recon: 137.4629 | KL: 6.4698
Epoch  12/25 | Loss: 142.9948 | Recon: 136.4324 | KL: 6.5624
Epoch  14/25 | Loss: 141.9910 | Recon: 135.3714 | KL: 6.6196
Epoch  16/25 | Loss: 141.3669 | Recon: 134.6940 | KL: 6.6729
Epoch  18/25 | Loss: 140.5310 | Recon: 133.7903 | KL: 6.7407
Epoch  20/25 | Loss: 140.0439 | Recon: 133.2640 | KL: 6.7799
Epoch  22/25 | Loss: 139.4625 | Recon: 132.6206 | KL: 6.8420
Epoch  24/25 | Loss: 138.9178 | Recon: 132.0557 | KL: 6.8621

Training Compl